# Training a Text Classification Model on DBpedia


## 1) Environment Setup (Local or Colab)



In [5]:
import os, sys

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = None

print("Current dir:", cwd)

if cwd is None or not os.path.exists(cwd):
    os.chdir('/content')
    print("Changed to /content")

print("cwd now", os.getcwd())

# then continue clone logic
is_colab = os.path.exists('/content')
if is_colab:
    target = '/content/text-classification'
    if os.path.exists(target):
        !rm -rf /content/text-classification
    !git clone https://github.com/huynhunguyen/text-classification.git /content/text-classification
    %cd /content/text-classification
    print('Running in Colab mode. Current dir:', os.getcwd())
else:
    # local mode
    print('Running in local mode. Current dir:', os.getcwd())

repo_root = os.getcwd()
sys.path.insert(0, os.path.join(repo_root, 'src'))
print('Repo root:', repo_root)
print('sys.path[0] =', sys.path[0])


Current dir: None
Changed to /content
cwd now /content
Cloning into '/content/text-classification'...
remote: Enumerating objects: 44, done.
remote: Total 44 (delta 0), reused 0 (delta 0), pack-reused 44 (from 1)
Receiving objects: 100% (44/44), 160.51 MiB | 47.45 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/text-classification
Running in Colab mode. Current dir: /content/text-classification
Repo root: /content/text-classification
sys.path[0] = /content/text-classification/src


## 2) Install dependencies


In [3]:
import os

# Remove old versions of PyTorch and related libraries to avoid conflicts
!pip uninstall -y torch torchvision torchaudio torchtext fastai timm || true

# Install a fully compatible set of versions
!pip install -q torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torchtext==0.18.0
!pip install -q "numpy>=1.26.0,<2.1"
!pip install -q datasets



Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
Found existing installation: fastai 2.8.6
Uninstalling fastai-2.8.6:
  Successfully uninstalled fastai-2.8.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 742.2 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 64.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 168.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 122.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 76.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 204.4 MB/s 

## 3) Mount Google Drive 


In [4]:
import os

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('Drive mounted: /content/drive')


Mounted at /content/drive
Drive mounted: /content/drive


## 4) Prepare dataset DBpedia


In [6]:
from datasets import load_dataset
import os


os.makedirs('data', exist_ok=True)
print("Downloading dataset from Hugging Face...")
dataset = load_dataset("dbpedia_14")

print("Saving dataset as CSV files...")
dataset['train'].to_csv('data/train.csv', index=False)
dataset['test'].to_csv('data/test.csv', index=False)

print("Completed! The data files are ready in the data directory.")
!ls -lh data

Đang tải dataset từ Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Đang lưu dataset thành tệp CSV...


Creating CSV from Arrow format:   0%|          | 0/560 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/70 [00:00<?, ?ba/s]

Hoàn tất! Các tệp dữ liệu đã sẵn sàng trong thư mục data/
total 185M
-rw-r--r-- 1 root root  146 Mar 21 04:18 classes.txt
-rw-r--r-- 1 root root 1.8K Mar 21 04:18 readme.txt
-rw-r--r-- 1 root root  21M Mar 21 04:20 test.csv
-rw-r--r-- 1 root root 164M Mar 21 04:20 train.csv


## 5) Training


In [8]:
%cd /content/text-classification
!python src/train.py

/content/text-classification
/usr/local/lib/python3.12/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.12/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.12/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS

## 6)  Backup model


In [10]:
!find models -maxdepth 3 -type f | head -n 20
!mkdir -p /content/drive/MyDrive/text_classification/models
!cp -r models/* /content/drive/MyDrive/text_classification/models


models/rnn/001.pth
models/transformer/004.pth
models/transformer/005.pth
models/transformer/003.pth
models/transformer/001.pth
models/transformer/002.pth


## 7) Predict

Execute the built-in inference script `src/predict.py` using a pre-trained checkpoint. The script will produce a predicted class label for the provided text.


In [8]:
%cd /content/text-classification
!python src/predict.py \
  --model transformer \
  --checkpoint models/transformer/005.pth \
  --input_text "The Academy for Environmental Leadership" \
  --max_len 128 \
  --embed_dim 128 \
  --num_heads 4 \
  --hidden_dim 256 \
  --num_layers 2

/content/text-classification
/usr/local/lib/python3.12/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
Traceback (most recent call last):
  File "/content/text-classification/src/predict.py", line 9, in <module>
    from src.models.rnn import RNNClassifier
ModuleNotFoundError: No module named 'src'
